# Due Diligence Agent — Deep Agents + Parallel

A research agent that reasons over its own confidence and chains follow-up queries when it isn't sure.

This notebook walks through the recipe end-to-end. The same code lives in `agent.py` so you can import it as a module or run it from the command line.

## Setup

```bash
uv venv
uv pip install -r requirements.txt
cp .env.example .env  # then fill in your keys
```

Make sure `ANTHROPIC_API_KEY` and `PARALLEL_API_KEY` are set in your environment before running the cells below. Get a Parallel key at [platform.parallel.ai](https://platform.parallel.ai).

In [ ]:
import os

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
assert os.environ.get("PARALLEL_API_KEY"), "PARALLEL_API_KEY not set"

## 1. The `research_task` tool

Every subagent calls a single tool that wraps Parallel's Task API and surfaces Basis metadata for the agent to reason over.

The wrapper does three things on top of `ParallelTaskRunTool`:

1. Routes the call through the SDK with the per-call output description.
2. Calls `parse_basis(result)` from `langchain-parallel` to extract `citations_by_field` and `low_confidence_fields`.
3. Surfaces a `low_confidence_warning` string when any field came back at low confidence, so the subagent can decide to chain a follow-up query using the returned `interaction_id`.

In [ ]:
from typing import Optional

from langchain_core.tools import tool
from langchain_parallel import ParallelTaskRunTool, parse_basis


@tool
def research_task(
    query: str,
    output_description: str,
    previous_interaction_id: Optional[str] = None,
) -> dict:
    """Run structured web research via Parallel's Task API.

    Returns findings with per-field citations and confidence scores (Basis).
    Use ``previous_interaction_id`` to chain a follow-up query that builds
    on a prior research session.
    """
    runner = ParallelTaskRunTool(
        processor="pro-fast",
        task_output_schema=output_description,
    )
    invoke_args: dict = {"input": query}
    if previous_interaction_id:
        invoke_args["previous_interaction_id"] = previous_interaction_id

    result = runner.invoke(invoke_args)
    parsed = parse_basis(result)

    output = result["output"]
    findings = output.get("content") if isinstance(output, dict) else output

    response: dict = {
        "findings": findings,
        "citations_by_field": parsed["citations_by_field"],
        "interaction_id": parsed["interaction_id"],
    }
    if parsed["low_confidence_fields"]:
        response["low_confidence_warning"] = (
            "These fields came back with low confidence and should be "
            "verified, ideally by chaining a follow-up query with "
            "previous_interaction_id: "
            + ", ".join(parsed["low_confidence_fields"])
        )
    return response

### Quick smoke test

Try the tool on a small structured query before wiring it into the agent. Expect a response with `findings`, `citations_by_field`, and `interaction_id`.

In [ ]:
result = research_task.invoke({
    "query": "Rivian Automotive founding year and current CEO",
    "output_description": "founding_year (int), current_ceo (str)",
})

print("findings:", result["findings"])
print("interaction_id:", result["interaction_id"])
print("fields with citations:", list(result.get("citations_by_field", {}).keys()))
if "low_confidence_warning" in result:
    print("warning:", result["low_confidence_warning"])

## 2. Define the subagents

Five tracks, each with a focused system prompt and the same `research_task` tool. Each subagent runs in an isolated context, so the orchestrator's window stays clean while each track can dig deep on its slice.

In [ ]:
from agent import (
    corporate_profile_subagent,
    financial_health_subagent,
    litigation_subagent,
    news_reputation_subagent,
    competitive_landscape_subagent,
)

for sa in [
    corporate_profile_subagent,
    financial_health_subagent,
    litigation_subagent,
    news_reputation_subagent,
    competitive_landscape_subagent,
]:
    print(f"- {sa['name']}: {sa['description']}")

## 3. Construct the orchestrator

The orchestrator uses Deep Agents' built-in `write_todos` planner, the `task` tool to dispatch subagents in parallel, the virtual filesystem to read each subagent's workpaper, and `ParallelWebSearchTool` for ad-hoc lookups when contradictions surface.

In [ ]:
from agent import agent

type(agent).__name__

## 4. Run a full DD on Rivian

Expect a ~15-30 minute run at the default `pro-fast` processor. The agent runs in three phases:

1. **Phase 1** — `write_todos` plans the diligence; the five subagents dispatch in parallel (corporate, financial, litigation, news, competitive-landscape).
2. **Phase 2 — competitor fan-out** — `competitive-landscape` returns 3 named competitors. The orchestrator dispatches a separate `competitor-analysis` subagent for **each** competitor, in parallel. This is the canonical Deep Agents pattern: spawning N instances of the same subagent type for N parallel investigations.
3. **Phase 3** — orchestrator reads every workpaper (5 Phase-1 files + 3 competitor files), cross-references contradictions, and synthesizes the final memo with a comparative competitor section.

Cost at default `pro-fast` tier: ~depends on processor pricing per run (~10-15 Task API calls including chained follow-ups and the 3 fan-out instances).

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Conduct a full due diligence report on Rivian Automotive."
    }]
})

print(result["messages"][-1].content)

## 5. Stream a run on a different target

For long runs, stream to see planning, tool calls, subagent activity, and the per-competitor fan-out in real time. `subgraphs=True` surfaces events from inside subagent execution.

In [ ]:
for chunk in agent.stream(
    {
        "messages": [{
            "role": "user",
            "content": "Conduct a full due diligence report on Anduril Industries."
        }]
    },
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    if chunk.get("type") == "updates":
        source = (
            f"[subagent: {chunk['ns']}]" if chunk.get("ns") else "[orchestrator]"
        )
        print(f"{source} {chunk.get('data')}")

## 6. Adapt the agent

The pattern doesn't care about the domain. Swap the five DD tracks for whatever you actually research:

- **Newsletter prep:** topic-survey / contrarian-views / primary-sources / open-questions subagents.
- **Lead generation:** company-overview / decision-maker / recent-signals / outreach-hook subagents.
- **Comparison shopping:** product-spec / price-and-availability / review-summary / shipping subagents.
- **Market sizing:** TAM / competitor-landscape / growth-drivers / regulatory-tailwinds subagents.
- **Candidate research:** background / public-work / network / red-flags subagents.

Each track is another subagent dict with a system prompt and the same `research_task` tool.